In [65]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision.models import densenet121
import numpy as np
import torch.nn.functional as F
from torch.utils.data import Dataset
from tqdm import tqdm
from recon_datasetv2 import ReconDataset
from torch.utils.data import DataLoader
import h5py

torch.cuda.is_available()

False

In [ ]:
with h5py.File('Data_SheppLogan/data.mat', 'r') as f:
    print(list(f.keys()))  # see variables inside
    
    # example: load a dataset
    N = f['N'][:]
    img_truth = f['img'][:]
    total_recon_curr = f['total_recon_curr'][:]
    total_recon_next = f['total_recon_next'][:]
    total_lambda = f['total_lambda'][:]

print('verifying shepplogan dataset')
print(total_recon_curr.shape)
print(total_recon_next.shape)
print(total_lambda.shape)

dataset_SheppLogan = ReconDataset(
    total_recon_curr,
    total_recon_next,
    total_lambda
)

loader_SheppLogan = DataLoader(
    dataset_SheppLogan,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)   

with h5py.File('Data_Cameraman/data.mat', 'r') as f:
    print(list(f.keys()))  # see variables inside
    
    # example: load a dataset
    N = f['N'][:]
    img_truth = f['img'][:]
    total_recon_curr = f['total_recon_curr'][:]
    total_recon_next = f['total_recon_next'][:]
    total_lambda = f['total_lambda'][:]

print('verifying cameraman dataset')
print(total_recon_curr.shape)
print(total_recon_next.shape)
print(total_lambda.shape)

dataset_Cameraman = ReconDataset(
    total_recon_curr,
    total_recon_next,
    total_lambda
)

loader_Cameraman = DataLoader(
    dataset_Cameraman,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)   

In [ ]:
class LambdaModel(nn.Module): 
    def __init__(self): 
        super().__init__() 
        base = densenet121(weights="IMAGENET1K_V1") 
        # Feature extractor (DenseNet backbone) 
        self.backbone = nn.Sequential( base.features, 
                                       nn.ReLU(inplace=True), 
                                       nn.AdaptiveAvgPool2d((1, 1)) ) 
        
        self.head = nn.Sequential( nn.Flatten(), 
                                   nn.Linear(1024, 128), 
                                   nn.ReLU(inplace=True), 
                                   nn.Linear(128, 64), 
                                   nn.ReLU(inplace=True), 
                                   nn.Linear(64, 1) ) 
        def forward(self, x): 
            # grayscale → 3-channel 
            if x.shape[1] == 1: 
                x = x.repeat(1, 3, 1, 1) 
            x = self.backbone(x) 
            lambda_raw = self.head(x) 
            # bound λ ∈ [0.05, 2] 
            # lambda_out = 0.05 + (2.0 - 0.05) * torch.sigmoid(lambda_raw) 
            lambda_out = F.softplus(lambda_raw) + 1e-3
        return lambda_out
        
class LambdaPrediction(nn.Module): 
    def __init__(self): 
        super().__init__() 
        self.lambdaModel = LambdaModel() 
    def forward(self, x_curr, x_next, lam_true): 
        lam_pred = self.lambdaModel(x_curr.repeat(1, 3, 1, 1)) 
        lam_true = lam_true.view(lam_true.size(0), 1, 1, 1) 
        direction = (x_next - x_curr) / (lam_true) 
        norm = torch.norm(direction.flatten(1), dim=1, keepdim=True) + 1e-6 
        direction = direction / norm.view(-1,1,1,1) 
        lam_pred = lam_pred.view(lam_pred.size(0), 1, 1, 1) 
        x_pred = x_curr + lam_pred * direction 
        loss = torch.mean((lam_pred - lam_true)**2) 
        
        return loss, lam_pred

In [66]:
# device = torch.device("cpu")
device = torch.device("mps")
# device = torch.device("cuda")

In [67]:
model = LambdaPrediction().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)

EPOCHS = 10

for epoch in range(EPOCHS):
    running_loss = 0.0

    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for i, (x_curr, x_next, lam_true) in enumerate(pbar):
        x_curr = x_curr.to(device)
        x_next = x_next.to(device)
        lam_true = lam_true.to(device)

        # forward
        loss_i, lam_pred = model(x_curr, x_next, lam_true)

        # backward
        optimizer.zero_grad()
        loss_i.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        # logging
        running_loss += loss_i.item()

        pbar.set_postfix(loss=loss_i.item())

        # if i % print_every == 0:
        #     print(f"\nBatch [{i}/{len(loader)}] Loss: {loss_i.item():.6f}")

    avg_loss = running_loss / len(loader)
    print(f"===> Epoch {epoch+1} Complete | Avg Loss: {avg_loss:.6f}")

Epoch 1/10: 100%|███████████████| 187/187 [00:44<00:00,  4.17it/s, loss=0.00207]


===> Epoch 1 Complete | Avg Loss: 0.022664


Epoch 2/10: 100%|███████████████| 187/187 [00:46<00:00,  4.04it/s, loss=0.00123]


===> Epoch 2 Complete | Avg Loss: 0.014134


Epoch 3/10: 100%|███████████████| 187/187 [00:45<00:00,  4.13it/s, loss=0.00124]


...

===> Epoch 8 Complete | Avg Loss: 0.002383


Epoch 9/10: 100%|███████████████| 187/187 [00:45<00:00,  4.08it/s, loss=0.00672]


===> Epoch 9 Complete | Avg Loss: 0.001936


Epoch 10/10: 100%|██████████████| 187/187 [00:45<00:00,  4.10it/s, loss=0.00107]

===> Epoch 10 Complete | Avg Loss: 0.002145


In [68]:
# testing model output
# v1 testing
model.eval()

x_curr, x_next, lam_true = next(iter(loader))

x_curr = x_curr.to(device)
x_next = x_next.to(device)
lam_true = lam_true.to(device)

with torch.no_grad():
    _, lam_pred = model(x_curr, x_next, lam_true)

print("\nV1:Sample-wise λ comparison:\n")
for i in range(5):
    print(f"Sample {i}:  pred = {lam_pred[i].item():.6f} | true = {lam_true[i].item():.6f}")

model.eval()

B, C, H, W = 8, 1, 64, 64  # 10 samples

x_zeros = torch.zeros((B, C, H, W)).to(device)
x_curr = x_zeros
x_next = x_zeros  # or you can also test random pairing

with torch.no_grad():
    _, lam_pred = model(x_curr, x_next, lam_true)

print("\nAll-zeros input λ predictions:\n")
for i in range(1):
    print(f"Sample {i}: {lam_pred[i].item():.6f}")

model.eval()

x_ones = torch.ones((B, C, H, W)).to(device)
x_curr = x_ones
x_next = x_ones

with torch.no_grad():
    _, lam_pred = model(x_curr, x_next, lam_true)

print("\nAll-ones input λ predictions:\n")
for i in range(1):
    print(f"Sample {i}: {lam_pred[i].item():.6f}")



V1:Sample-wise λ comparison:

Sample 0:  pred = 0.014912 | true = 1.350000
Sample 1:  pred = 0.037197 | true = 1.650000
Sample 2:  pred = 0.038185 | true = 1.700000
Sample 3:  pred = 0.018179 | true = 1.350000
Sample 4:  pred = 1.682866 | true = 1.750000

All-zeros input λ predictions:

Sample 0: 33.040112

All-ones input λ predictions:

Sample 0: 34.303989


In [52]:

class LambdaModelV3(nn.Module):
    def __init__(self):
        super().__init__()

        # x_curr (1) + residual (1) + residual_norm (1) = 3 channels
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.GroupNorm(8, 32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.GroupNorm(8, 64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.GroupNorm(8, 128),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d(1)
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x_curr, residual):
        B, _, H, W = x_curr.shape

        # residual magnitude → broadcast
        res_norm = torch.norm(residual.flatten(1), dim=1, keepdim=True)
        res_norm = res_norm.view(B, 1, 1, 1).expand(B, 1, H, W)

        x = torch.cat([x_curr, residual, res_norm], dim=1)

        h = self.backbone(x)
        lam = self.head(h)

        # positive λ, no saturation
        lam = F.softplus(lam) + 1e-3
        return lam
class LambdaPredictionV3(nn.Module):
    def __init__(self):
        super().__init__()
        self.lambdaModel = LambdaModelV3()

    def forward(self, x_curr, x_next, lam_true=None):

        # raw residual (NO normalization)
        residual = x_next - x_curr

        lam_pred = self.lambdaModel(x_curr, residual)
        lam_pred = lam_pred.view(-1, 1, 1, 1)

        # predicted update
        x_pred = x_curr + lam_pred * residual

        # --- weighted reconstruction loss ---
        weight = torch.norm(residual.flatten(1), dim=1)  # (B,)

        recon = (x_pred - x_next) ** 2
        recon = recon.mean(dim=(1, 2, 3))  # per-sample

        loss = (weight * recon).mean()

        # optional λ supervision
        if lam_true is not None:
            lam_true = lam_true.view(-1, 1, 1, 1)
            lam_loss = F.mse_loss(lam_pred, lam_true)
            loss = loss + 0.01 * lam_loss

        # anti-collapse regularization
        lam_var = lam_pred.view(lam_pred.size(0), -1).var(dim=0).mean()
        loss = loss - 0.01 * lam_var

        return loss, lam_pred

In [53]:
model_v3 = LambdaPredictionV3().to(device)

optimizer = torch.optim.AdamW(
    model_v3.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

EPOCHS = 10

for epoch in range(EPOCHS):
    model_v3.train()
    running_loss = 0

    pbar = tqdm(loader, desc=f"[V2] Epoch {epoch+1}/{EPOCHS}")

    for x_curr, x_next, lam_true in pbar:

        x_curr = x_curr.to(device)
        x_next = x_next.to(device)
        lam_true = lam_true.to(device)

        loss, lam_pred = model_v3(x_curr, x_next, lam_true)

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model_v2.parameters(), 1.0)

        optimizer.step()

        running_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    print(f"[V2] Epoch {epoch+1}: avg loss = {running_loss / len(loader):.6f}")

[V2] Epoch 1/10: 100%|█████████| 187/187 [00:06<00:00, 28.64it/s, loss=0.000513]


[V2] Epoch 1: avg loss = 0.001054


[V2] Epoch 2/10: 100%|██████████| 187/187 [00:03<00:00, 50.56it/s, loss=9.09e-5]


[V2] Epoch 2: avg loss = 0.000147


[V2] Epoch 3/10: 100%|█████████| 187/187 [00:03<00:00, 50.53it/s, loss=0.000166]


...

[V2] Epoch 8: avg loss = -0.001699


[V2] Epoch 9/10: 100%|█████████| 187/187 [00:03<00:00, 47.18it/s, loss=0.000967]


[V2] Epoch 9: avg loss = -0.000462


[V2] Epoch 10/10: 100%|█████████| 187/187 [00:03<00:00, 46.86it/s, loss=-0.0117]

[V2] Epoch 10: avg loss = -0.001374



V1:Sample-wise λ comparison:

Sample 0:  pred = 1.558455 | true = 1.650000
Sample 1:  pred = 1.557072 | true = 1.650000
Sample 2:  pred = 1.563669 | true = 1.700000
Sample 3:  pred = 1.563765 | true = 1.600000
Sample 4:  pred = 1.561053 | true = 1.700000

V3: Sample-wise λ comparison:

Sample 0:  pred = 0.708504 | true = 1.600000
Sample 1:  pred = 0.460947 | true = 1.600000
Sample 2:  pred = 0.461961 | true = 1.650000
Sample 3:  pred = 0.462590 | true = 1.350000
Sample 4:  pred = 3.574685 | true = 1.550000


In [ ]:
# redo with new data
with h5py.File('Data3/data.mat', 'r') as f:
    print(list(f.keys()))  # see variables inside
    
    # example: load a dataset
    N = f['N'][:]
    img_truth = f['img'][:]
    total_recon_curr = f['total_recon_curr'][:]
    total_recon_next = f['total_recon_next'][:]
    total_lambda = f['total_lambda'][:]
print(total_recon_curr.shape)
print(total_recon_next.shape)
print(total_lambda.shape)

In [ ]:
# retry model 1 with new data
model = LambdaPrediction().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)

EPOCHS = 10
print_every = 50

for epoch in range(EPOCHS):
    running_loss = 0.0

    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for i, (x_curr, x_next, lam_true) in enumerate(pbar):
        x_curr = x_curr.to(device)
        x_next = x_next.to(device)
        lam_true = lam_true.to(device)

        # forward
        loss_i, lam_pred = model(x_curr, x_next, lam_true)

        # backward
        optimizer.zero_grad()
        loss_i.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        # logging
        running_loss += loss_i.item()

        pbar.set_postfix(loss=loss_i.item())

        # if i % print_every == 0:
        #     print(f"\nBatch [{i}/{len(loader)}] Loss: {loss_i.item():.6f}")

    avg_loss = running_loss / len(loader)
    print(f"===> Epoch {epoch+1} Complete | Avg Loss: {avg_loss:.6f}")

In [ ]:
model_v3 = LambdaPredictionV3().to(device)

optimizer = torch.optim.AdamW(
    model_v3.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

EPOCHS = 10

for epoch in range(EPOCHS):
    model_v3.train()
    running_loss = 0

    pbar = tqdm(loader, desc=f"[V2] Epoch {epoch+1}/{EPOCHS}")

    for x_curr, x_next, lam_true in pbar:

        x_curr = x_curr.to(device)
        x_next = x_next.to(device)
        lam_true = lam_true.to(device)

        loss, lam_pred = model_v2(x_curr, x_next, lam_true)

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model_v2.parameters(), 1.0)

        optimizer.step()

        running_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    print(f"[V2] Epoch {epoch+1}: avg loss = {running_loss / len(loader):.6f}")

In [ ]:
# testing model output
# v1 testing
model.eval()

x_curr, x_next, lam_true = next(iter(loader))

x_curr = x_curr.to(device)
x_next = x_next.to(device)
lam_true = lam_true.to(device)

with torch.no_grad():
    _, lam_pred = model_v2(x_curr, x_next)

print("\nV1:Sample-wise λ comparison:\n")
for i in range(10):
    print(f"Sample {i}:  pred = {lam_pred[i].item():.6f} | true = {lam_true[i].item():.6f}")

# v2 testing
model_v2.eval()

x_curr, x_next, lam_true = next(iter(loader))

x_curr = x_curr.to(device)
x_next = x_next.to(device)
lam_true = lam_true.to(device)

with torch.no_grad():
    _, lam_pred = model_v2(x_curr, x_next)

print("\nV2: Sample-wise λ comparison:\n")
for i in range(10):
    print(f"Sample {i}:  pred = {lam_pred[i].item():.6f} | true = {lam_true[i].item():.6f}")